# System Looting — free public server (Google Colab + playit.gg)

Runs the **System Looting** Luanti server on Google's free Colab machines and
exposes it to the internet through a **playit.gg** tunnel, so anyone can join
from a normal Luanti client. **No credit card needed** — just a Google account.

**Limits (free tier):** sessions run up to ~12 h, disconnect after ~90 min of
inactivity, and the VM is wiped afterwards. Great for play sessions, not for a
24/7 server (see `docs/FREE_HOSTING.md` for Oracle free tier etc.).

**How:** Runtime → *Run all*. Then follow the instructions printed at the end
(one-time playit account claim).

In [ ]:
# 1) Install the Luanti engine (server-only binary from Ubuntu's repos)
import subprocess, shutil
subprocess.run('sudo apt-get update -qq && sudo apt-get install -y -qq minetest-server || '
               'sudo apt-get install -y -qq luanti-server', shell=True, timeout=600)
engine = shutil.which('luantiserver') or shutil.which('minetestserver')
assert engine, 'No engine binary found — rerun this cell.'
print('engine:', engine)
print(subprocess.run(f'{engine} --version', shell=True, capture_output=True, text=True).stdout.splitlines()[0])

In [ ]:
# 2) Get the game, create the world, start the server on UDP 30000
import subprocess, shutil, time, os
os.makedirs('/content/game', exist_ok=True)
if not os.path.exists('/content/game/game.conf'):
    subprocess.run('git clone --depth 1 https://github.com/SodoMita/SystemTest /content/game',
                   shell=True, check=True, timeout=300)
os.makedirs('/content/game/worlds/systemloot', exist_ok=True)
open('/content/game/worlds/systemloot/world.mt', 'w').write(
    'gameid = SystemTest\nbackend = sqlite3\nmg_name = singlenode\n')
open('/content/game/systemloot.conf', 'w').write('''
server_name = System Looting — Colab
server_description = Free Colab test server
port = 30000
bind_address = 0.0.0.0
max_users = 16
mg_name = singlenode
time_speed = 0
enable_damage = true
sl_auto_start = true
sl_auto_start_delay = 20
''')
engine = shutil.which('luantiserver') or shutil.which('minetestserver')
subprocess.Popen(f'{engine} --gameid SystemTest --world /content/game/worlds/systemloot '
                 f'--config /content/game/systemloot.conf --logfile /content/game/server.log',
                 shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
log = ''
for _ in range(20):
    time.sleep(1)
    log = open('/content/game/server.log').read()
    if 'listening on' in log: break
print('server up:', 'listening on' in log)
print('\n'.join(l for l in log.splitlines() if 'listening' in l or 'Loaded core' in l or 'arena' in l))

In [ ]:
# 3) playit.gg tunnel → public address
import subprocess, time, os, re
if not os.path.exists('/content/playit'):
    subprocess.run('curl -SsL https://playit.gg/download/playit-linux-amd64 -o /content/playit && chmod +x /content/playit',
                   shell=True, check=True, timeout=300)
subprocess.Popen('script -q -c /content/playit /content/playit.log', shell=True,
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(12)
log = open('/content/playit.log', errors='replace').read()
claim = re.search(r'https://playit\.gg/claim/[A-Za-z0-9]+', log)
print('=' * 66)
print('  PUBLIC SERVER — NEXT STEPS (once, ~2 minutes):')
print('=' * 66)
if claim:
    print('  1. OPEN THIS CLAIM LINK in any browser:')
    print('     ' + claim.group(0))
    print('     (create the free playit account — no card needed)')
else:
    print('  1. Run:  cat /content/playit.log   and look for the claim URL.')
print('  2. After claiming: playit.gg dashboard → add tunnel:')
print('     Type UDP, local port 30000  (and one TCP 30000)')
print('  3. Share the public address playit shows (e.g. 123.45.67.89:51234).')
print('  4. Server log any time:  tail /content/game/server.log')
print('=' * 66)

### After claiming

1. playit.gg dashboard → add tunnel: **UDP**, local port **30000** (and one **TCP** 30000).
2. playit shows a public address like `123.45.67.89:51234` — share it; anyone joins from Luanti.
3. Keep this tab open; interact occasionally (idle >90 min disconnects; sessions end ~12 h — rerun *Run all*; world resets unless backed up, see next cell).

For a permanent free server: `docs/FREE_HOSTING.md` (Oracle Always Free).

In [ ]:
# Optional: back up the world to Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# import subprocess, os
# os.makedirs('/content/drive/MyDrive/systemloot', exist_ok=True)
# subprocess.run('cp -r /content/game/worlds/systemloot /content/drive/MyDrive/systemloot/', shell=True)
# print('world backed up')